In [3]:
conda install -c conda-forge xgboost

Solving environment: done


==> WARNING: A newer version of conda exists. <==
  current version: 23.7.4
  latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda

Or to minimize the number of packages updated during conda update use

     conda install conda=26.1.1



## Package Plan ##

  environment location: /Users/viki/anaconda3

  added / updated specs:
    - xgboost


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    libcxx-22.1.0              |       h55c6f16_1         556 KB  conda-forge
    ------------------------------------------------------------
                                           Total:         556 KB

The following NEW packages will be INSTALLED:

  libxgboost         conda-forge/osx-arm64::libxgboost-3.2.0-cpu_h77aee5e_1 
  py-xgboost         conda-forge/noarch::py-xgboost-3.2.0-cpu_pyh718b53a_1 
  xgboost            cond

In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


# -----------------------------------------
# STEP 1 — Load Final Dataset
# -----------------------------------------

project_root = os.path.abspath("..")
data_folder = os.path.join(project_root, "data")

final_path = os.path.join(data_folder, "final_spatial_irrigation_dataset.csv")
df = pd.read_csv(final_path)

print("Initial dataset shape:", df.shape)


# -----------------------------------------
# STEP 2 — FIX DATA TYPE ISSUES
# -----------------------------------------

# Fix Water_Liters if stored like "[1.0647519E5]"
df["Water_Liters"] = (
    df["Water_Liters"]
    .astype(str)
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .astype(float)
)

# Ensure numeric columns are numeric
numeric_cols = ["Latitude", "Longitude", "Temperature", "Humidity", "Rainfall"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop any rows that failed conversion
df = df.dropna()

print("Cleaned dataset shape:", df.shape)


# -----------------------------------------
# STEP 3 — Encode Categorical Features
# -----------------------------------------

df = pd.get_dummies(df, columns=["Season", "Soil_Type"], drop_first=True)

print("Encoded feature count:", df.shape[1])


# -----------------------------------------
# STEP 4 — Define Features and Target
# -----------------------------------------

X = df.drop(["Date", "Water_Liters"], axis=1)
y = df["Water_Liters"]

print("Final feature count:", X.shape[1])


# -----------------------------------------
# STEP 5 — Train/Test Split
# -----------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# -----------------------------------------
# STEP 6 — Define Models
# -----------------------------------------

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(n_estimators=100),
    "XGBoost": XGBRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        random_state=42
    )
}

results = {}


# -----------------------------------------
# STEP 7 — Train & Evaluate
# -----------------------------------------

for name, model in models.items():
    
    print(f"\nTraining {name}...")
    
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    cv_scores = cross_val_score(model, X, y, cv=5, scoring="r2")
    
    results[name] = {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "CV_R2_Mean": cv_scores.mean()
    }


# -----------------------------------------
# STEP 8 — Show Model Comparison
# -----------------------------------------

results_df = pd.DataFrame(results).T
print("\nModel Comparison:")
print(results_df)


# -----------------------------------------
# STEP 9 — Save Best Model (XGBoost)
# -----------------------------------------

best_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

best_model.fit(X_train, y_train)

model_folder = os.path.join(project_root, "models")
os.makedirs(model_folder, exist_ok=True)

model_path = os.path.join(model_folder, "best_spatial_model.pkl")
joblib.dump(best_model, model_path)

print("\n✅ Best model saved at:", model_path)

Initial dataset shape: (54810, 9)
Cleaned dataset shape: (54810, 9)
Encoded feature count: 14
Final feature count: 12

Training Linear Regression...

Training Decision Tree...

Training Random Forest...

Training XGBoost...

Model Comparison:
                            MAE          RMSE        R2  CV_R2_Mean
Linear Regression  16781.578935  25817.517225  0.741584    0.727450
Decision Tree        593.233803   1148.023610  0.999489    0.994705
Random Forest        279.073180    614.180023  0.999854    0.995121
XGBoost              521.216580    792.537414  0.999756    0.997328

✅ Best model saved at: /Users/viki/Desktop/smart-irrigation-xai/models/best_spatial_model.pkl


In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


In [2]:
# -----------------------------------------
# STEP 1 — Load Final Dataset
# -----------------------------------------

project_root = os.path.abspath("..")
data_folder = os.path.join(project_root, "data")

final_path = os.path.join(data_folder, "final_spatial_irrigation_dataset.csv")
df = pd.read_csv(final_path)

print("Initial dataset shape:", df.shape)

Initial dataset shape: (54810, 9)


In [3]:

# -----------------------------------------
# STEP 2 — FIX DATA TYPE ISSUES
# -----------------------------------------

# Fix Water_Liters if stored like "[1.0647519E5]"
df["Water_Liters"] = (
    df["Water_Liters"]
    .astype(str)
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .astype(float)
)

In [4]:
# Ensure numeric columns are numeric
numeric_cols = ["Latitude", "Longitude", "Temperature", "Humidity", "Rainfall"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop any rows that failed conversion
df = df.dropna()

print("Cleaned dataset shape:", df.shape)



Cleaned dataset shape: (54810, 9)


In [5]:
# -----------------------------------------
# STEP 3 — Encode Categorical Features
# -----------------------------------------

df = pd.get_dummies(df, columns=["Season", "Soil_Type"], drop_first=True)

print("Encoded feature count:", df.shape[1])

Encoded feature count: 14


In [6]:
# -----------------------------------------
# STEP 4 — Define Features and Target
# -----------------------------------------

X = df.drop(["Date", "Water_Liters"], axis=1)
y = df["Water_Liters"]

print("Final feature count:", X.shape[1])


Final feature count: 12


In [7]:
# -----------------------------------------
# STEP 5 — Train/Test Split
# -----------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
# -----------------------------------------
# STEP 6 — Define Models
# -----------------------------------------

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(n_estimators=100),
    "XGBoost": XGBRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        random_state=42
    )
}

results = {}


In [9]:

# -----------------------------------------
# STEP 7 — Train & Evaluate
# -----------------------------------------

for name, model in models.items():
    
    print(f"\nTraining {name}...")
    
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    cv_scores = cross_val_score(model, X, y, cv=5, scoring="r2")
    
    results[name] = {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "CV_R2_Mean": cv_scores.mean()
    }


Training Linear Regression...

Training Decision Tree...

Training Random Forest...

Training XGBoost...


In [10]:
# -----------------------------------------
# STEP 8 — Show Model Comparison
# -----------------------------------------

results_df = pd.DataFrame(results).T
print("\nModel Comparison:")
print(results_df)


Model Comparison:
                            MAE          RMSE        R2  CV_R2_Mean
Linear Regression  16781.578935  25817.517225  0.741584    0.727450
Decision Tree        591.970905   1138.586674  0.999497    0.994000
Random Forest        277.726566    611.008601  0.999855    0.995019
XGBoost              521.216580    792.537414  0.999756    0.997328


In [11]:
# -----------------------------------------
# STEP 9 — Save Best Model (XGBoost)
# -----------------------------------------

best_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

best_model.fit(X_train, y_train)

model_folder = os.path.join(project_root, "models")
os.makedirs(model_folder, exist_ok=True)

model_path = os.path.join(model_folder, "best_spatial_model.pkl")
joblib.dump(best_model, model_path)

print("\n✅ Best model saved at:", model_path)


✅ Best model saved at: /Users/viki/Desktop/smart-irrigation-xai/models/best_spatial_model.pkl
